In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import urllib.request
import os

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")


Using device: mps


In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
filename = "shakespeare.txt"

if not os.path.exists(filename):
    print(f"Downloading from {url}...")
    urllib.request.urlretrieve(url, filename)
    print("Downloaded.")
else:
    print("Already have the file.")

with open(filename, "r") as f:
    text = f.read()

print(f"Total characters: {len(text):,}")
print(f"\nFirst 300 characters:\n{text[:300]}")


Already have the file.
Total characters: 1,115,394

First 300 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [5]:
# Find all unique characters in the text
chars = sorted(set(text))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

# Create mappings between characters and integer indices
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}
print(f"\nCharacter to index mapping: {char_to_idx}")
print(f"Index to character mapping: {idx_to_char}")

# Test the mappings
sample = "Hello"
encoded = [char_to_idx[c] for c in sample]
decoded = ''.join(idx_to_char[i] for i in encoded)
print(f"\n'{sample}' -> {encoded} --> '{decoded}'")



Vocabulary size: 65
Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Character to index mapping: {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
Index to character mapping: {0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R

In [12]:
# Encode the entire text as a tensor of integers
data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
print(f"Encoded data shape: {data.shape}")
print(f"First 50 tokens: {data[:50]}")

# Train/test split: 90/10
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]
print(f"\nTrain size: {len(train_data):,}")
print(f"\nTest size: {len(test_data)}")


Encoded data shape: torch.Size([1115394])
First 50 tokens: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])

Train size: 1,003,854

Test size: 111540


In [15]:
# Length of each input sequence (context window)
block_size = 64
# How many sequences per batch
batch_size = 32

def get_batch(split):
    """Sample a batch of sequences from the dataset."""
    source = train_data if split == 'train' else test_data
    # Pick batch_size random starting positions
    ix = torch.randint(0, len(source) - block_size - 1, (batch_size,))
    # For each starting position, grab block_size characters as input...
    x = torch.stack([source[i:i+block_size] for i in ix])
    # ... and the next character at each position as target
    y = torch.stack([source[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# Test it
xb, yb = get_batch('train')
print(f"Input batch shape: {xb.shape}")
print(f"Target batch shape: {yb.shape}")
print(f"\nFirst sequence input: {repr(''.join(idx_to_char[i.item()] for i in xb[0]))}")
print(f"First sequence target: {repr(''.join(idx_to_char[i.item()] for i in yb[0]))}")
    

Input batch shape: torch.Size([32, 64])
Target batch shape: torch.Size([32, 64])

First sequence input: " with a\nwhite wench's black eye; shot through the ear with a\nlov"
First sequence target: "with a\nwhite wench's black eye; shot through the ear with a\nlove"


In [18]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128, num_layers=2):
        super().__init__()
        # Embedding: turn integer character IDs into dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # LSTM: process the sequence with recurrent state
        self.lstm = nn.LSTM(
            input_size = embedding_dim,
            hidden_size = hidden_dim,
            num_layers = num_layers,
            batch_first = True, # Expect [batch, seq, features] not [seq, batch, features]
        )
        # Output projection: hidden state -> vocab logits
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        # x: [batch, seq_len] integers
        emb = self.embedding(x)                 # [batch, seq_len, embedding_dim]
        out, hidden = self.lstm(emb, hidden)    # [batch, seq_len, hidden_dim]
        logits = self.fc(out)                   # [batch, seq_len, vocab_size]
        return logits, hidden

model = CharRNN(vocab_size).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

CharRNN(
  (embedding): Embedding(65, 64)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=65, bias=True)
)

Total parameters: 243,969


In [20]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
num_iterations = 2000
eval_interval = 200

train_losses = []
test_lossses = []

@torch.no_grad()
def estimate_loss():
    """Compute average loss on a few batches, for both train and test."""
    model.eval()
    losses = {}
    for split in ["train", "test"]:
        batch_losses = []
        for _ in range(20):
            x, y = get_batch(split)
            logits, _ = model(x)
            # Reshape for loss: [batch*seq, vocab] vs [batch*seq]
            loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))
            batch_losses.append(loss.item())
        losses[split] = sum(batch_losses) / len(batch_losses)
    model.train()
    return losses

model.train()
for it in range(num_iterations):
    # Get a batch of data
    x, y = get_batch('train')

    # Forward pass
    logits, _ = model(x)

    # Compute loss: reshape to [batch*seq, vocab] vs [batch*seq]
    loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))

    # Backward pass and optimization step
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Logging
    if it % eval_interval == 0:
        losses = estimate_loss()
        train_losses.append(losses['train'])
        test_lossses.append(losses['test'])
        print(f"Step {it:4d} | train loss: {losses['train']:.4f} | test loss: {losses['test']:.4f}")
print("\nTraining complete.")